In [13]:
!pip install fastapi uvicorn[standard] pydantic httpx nest-asyncio -q

# Phase 4 — Deploy & Ship
## Day 25: FastAPI — Wrap Your RAG Pipeline in a Real API

**Goal:** Turn the Indian Legal RAG chatbot from a Gradio UI into a proper REST API
with documented endpoints, type-safe request/response models, and Swagger UI.

**Today's endpoints:**
- `GET  /health`   — status check
- `POST /chat`     — question → grounded answer + sources
- `POST /predict`  — text → sentiment label + confidence

**Stack:** FastAPI + Pydantic + Uvicorn

## Part 1: What Makes FastAPI Different

FastAPI sits on top of two libraries:
- **Starlette** — handles HTTP routing, requests, responses (the web layer)
- **Pydantic** — handles data validation (the safety layer)

The combination means: you define *what* valid data looks like,
and FastAPI enforces it *before* your code runs.

Compare Flask (manual everything) vs FastAPI (declarative, type-driven).

In [14]:
from pydantic import BaseModel
from typing import Optional, List

class ChatRequest(BaseModel):
    question: str
    session_id: Optional[str] = None

class SourceChunk(BaseModel):
    content: str
    source: str
    page: Optional[int] = None

class ChatResponse(BaseModel):
    answer: str
    sources: List[SourceChunk]
    session_id: str

# ── Valid input ───────────────────────────────────────────────────────────────
valid = ChatRequest(question="What is BNS Section 103?")
print("Valid:", valid)
print("question type:", type(valid.question))
print("session_id:", valid.session_id)

print()

# ── Pydantic v2: int → str is NOT coerced (strict by default for str) ─────────
print("--- Pydantic v2 strict str behavior ---")
try:
    coerced = ChatRequest(question=42)
except Exception as e:
    print(f"int → str: REJECTED ({type(e).__name__})")
    print("  Reason: Pydantic v2 does NOT silently coerce int to str")

print()

# ── What Pydantic v2 DOES coerce ──────────────────────────────────────────────
print("--- What v2 DOES coerce ---")

class FlexibleModel(BaseModel):
    count: int      # str "5" → int 5 : YES coerced
    score: float    # int 3 → float 3.0 : YES coerced

f1 = FlexibleModel(count="5", score=3)  # both coerced silently
print(f"str '5' → int: {f1.count}  (type: {type(f1.count).__name__})")
print(f"int 3 → float: {f1.score}  (type: {type(f1.score).__name__})")

print()

# ── opt-in coercion for str using model_config ─────────────────────────────────
print("--- Opt-in str coercion (if you explicitly want it) ---")
from pydantic import ConfigDict

class CoercingModel(BaseModel):
    model_config = ConfigDict(coerce_numbers_to_str=True)
    question: str

c = CoercingModel(question=42)
print(f"int 42 → str with coerce_numbers_to_str=True: '{c.question}'")

print()

# ── Missing required field — still caught ─────────────────────────────────────
print("--- Missing required field ---")
try:
    broken = ChatRequest()
except Exception as e:
    print(f"Missing field: REJECTED ({type(e).__name__})")
    print(f"  Detail: {e.errors()[0]['msg']}")

Valid: question='What is BNS Section 103?' session_id=None
question type: <class 'str'>
session_id: None

--- Pydantic v2 strict str behavior ---
int → str: REJECTED (ValidationError)
  Reason: Pydantic v2 does NOT silently coerce int to str

--- What v2 DOES coerce ---
str '5' → int: 5  (type: int)
int 3 → float: 3.0  (type: float)

--- Opt-in str coercion (if you explicitly want it) ---
int 42 → str with coerce_numbers_to_str=True: '42'

--- Missing required field ---
Missing field: REJECTED (ValidationError)
  Detail: Field required


## Part 2: Build the FastAPI App — Step by Step

Three layers:
1. **App instance** — the FastAPI object with metadata
2. **Startup event** — load the RAG pipeline *once*, store it in app state
3. **Endpoints** — thin functions that call the pipeline and return typed responses

The startup event is critical. Loading a model takes seconds.
You load it once when the server starts — not on every request.

In [15]:
# %%writefile app/main.py
# We write this to a file — this IS the production code, not just a demo

app_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional, List
import os
import uuid

# ── App Metadata ─────────────────────────────────────────────────────────────
app = FastAPI(
    title="Indian Legal RAG API",
    description="""
    REST API wrapping a Retrieval-Augmented Generation pipeline 
    over the Bharatiya Nyaya Sanhita (BNS) 2023.
    
    - POST /chat: Ask a legal question, get a grounded answer with source citations
    - POST /predict: Run sentiment classification on any text
    - GET  /health: Check if the API is alive
    """,
    version="1.0.0",
    contact={"name": "Faisal Imam", "url": "https://github.com/faisalimam1"},
)

# ── Pydantic Models — Request/Response Contracts ──────────────────────────────
class ChatRequest(BaseModel):
    question: str
    session_id: Optional[str] = None       # client passes this to maintain context

class SourceChunk(BaseModel):
    content: str                            # the actual retrieved text chunk
    source: str                             # document name / section
    relevance_score: Optional[float] = None

class ChatResponse(BaseModel):
    answer: str
    sources: List[SourceChunk]
    session_id: str                         # always returned so client can track

class PredictRequest(BaseModel):
    text: str

class PredictResponse(BaseModel):
    label: str                              # "POSITIVE" or "NEGATIVE"
    confidence: float                       # 0.0 → 1.0
    model_used: str

class HealthResponse(BaseModel):
    status: str
    pipeline_loaded: bool
    version: str

# ── In-memory session store (replace with Redis in production) ────────────────
# Maps session_id → list of (question, answer) tuples
session_store: dict = {}

# ── Pipeline placeholder (replace with your actual RAG pipeline) ──────────────
# In production: load ChromaDB, BM25 index, Groq client at startup
# For this demo: we return structured mock responses

def get_rag_answer(question: str, session_id: str) -> tuple[str, List[dict]]:
    """
    In production this calls your Phase 3 RAG pipeline:
        1. Hybrid retrieval (BM25 + semantic, RRF fusion)
        2. Context assembly
        3. Groq LLM call with retrieved context
        4. Return answer + source chunks
    
    For notebook demo: returns structured mock data.
    """
    history = session_store.get(session_id, [])
    
    # Mock response — replace this block with actual pipeline call
    mock_answer = (
        f"Based on the Bharatiya Nyaya Sanhita 2023, regarding '{question}': "
        f"[This is where your LangChain LCEL chain returns the grounded answer. "
        f"Context from {len(history)} prior turns available.]"
    )
    mock_sources = [
        {"content": "BNS Section 103 — Murder...", "source": "BNS_2023.pdf", "relevance_score": 0.91},
        {"content": "BNS Section 104 — Culpable homicide...", "source": "BNS_2023.pdf", "relevance_score": 0.87},
    ]
    
    # Store in session
    session_store[session_id] = history + [(question, mock_answer)]
    
    return mock_answer, mock_sources


def get_sentiment(text: str) -> dict:
    """
    In production: loads your fine-tuned BERT from HF Hub.
    For demo: returns mock sentiment.
    """
    return {
        "label": "POSITIVE" if len(text) % 2 == 0 else "NEGATIVE",
        "confidence": 0.94,
        "model_used": "faisalimam19/bert-imdb-sentiment"
    }


# ── Endpoints ──────────────────────────────────────────────────────────────────

@app.get("/health", response_model=HealthResponse)
def health_check():
    """Check if the API is alive and the pipeline is loaded."""
    return HealthResponse(
        status="healthy",
        pipeline_loaded=True,
        version="1.0.0"
    )


@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    """
    Ask a legal question. Returns a grounded answer with source citations.
    
    - Pass session_id to maintain conversation context across turns.
    - If session_id is None, a new session is created and returned.
    """
    if not request.question.strip():
        raise HTTPException(status_code=422, detail="question cannot be empty")
    
    # Generate session ID if client didn't provide one
    session_id = request.session_id or str(uuid.uuid4())
    
    try:
        answer, raw_sources = get_rag_answer(request.question, session_id)
    except Exception as e:
        # Never expose internal errors to the client
        raise HTTPException(status_code=500, detail="RAG pipeline error. Check server logs.")
    
    sources = [SourceChunk(**s) for s in raw_sources]
    
    return ChatResponse(
        answer=answer,
        sources=sources,
        session_id=session_id
    )


@app.post("/predict", response_model=PredictResponse)
async def predict(request: PredictRequest):
    """
    Run sentiment classification on input text.
    Uses the fine-tuned BERT model (F1: 0.921 on IMDB).
    """
    if len(request.text.strip()) < 3:
        raise HTTPException(status_code=422, detail="text too short for classification")
    
    try:
        result = get_sentiment(request.text)
    except Exception as e:
        raise HTTPException(status_code=500, detail="Model inference failed. Check server logs.")
    
    return PredictResponse(**result)
'''

# Write to file
import os
os.makedirs("app", exist_ok=True)

with open("app/main.py", "w") as f:
    f.write(app_code)

print("✅ app/main.py written")
print(f"   Lines: {len(app_code.splitlines())}")

✅ app/main.py written
   Lines: 155


## Part 3: Run the Server and Test It

**How FastAPI runs:**
- `uvicorn` is the ASGI server — it receives HTTP requests and passes them to FastAPI
- `app.main:app` → "in the file app/main.py, find the object called app"
- `--reload` → restart server when code changes (dev mode only, never in production)

**In production you'd run:**
```bash
uvicorn app.main:app --host 0.0.0.0 --port 8000
```

**Here in the notebook**, we run it in a background thread so we can 
test it from the same kernel using the `requests` library.

In [16]:
import threading
import time
import nest_asyncio

# FastAPI uses async — Kaggle's kernel is also async — this lets them coexist
nest_asyncio.apply()

def run_server():
    import uvicorn
    # Import the app from the file we just wrote
    import sys
    sys.path.insert(0, ".")
    exec(open("app/main.py").read(), globals())
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Give the server 2 seconds to start
time.sleep(2)
print("✅ Server started at http://localhost:8000")
print("   Swagger UI: http://localhost:8000/docs")

/usr/local/lib/python3.12/dist-packages/uvicorn/server.py:75: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
Exception in thread Thread-7 (run_server):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_59/3793162754.py", line 14, in run_server
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/main.py", line 606, in run
    server.run()
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 75, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: _patch_asyncio.<locals>.run() got an unexpected keyword 

✅ Server started at http://localhost:8000
   Swagger UI: http://localhost:8000/docs


In [18]:
# ── Cell 1: Start server — no nest_asyncio needed ────────────────────────────
import asyncio
import uvicorn
import threading
import time
import requests
import json
import sys

sys.path.insert(0, ".")
from app.main import app

def run_server():
    # Create a fresh event loop in this thread — completely isolated from Kaggle's kernel loop
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
    server = uvicorn.Server(config)
    
    loop.run_until_complete(server.serve())

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(3)

try:
    ping = requests.get("http://localhost:8000/health", timeout=3)
    print(f"✅ Server is UP — {ping.status_code}")
    print(f"   {ping.json()}")
except Exception as e:
    print(f"❌ Still failed: {e}")
    print("   → Try Option B below")

✅ Server is UP — 200
   {'status': 'healthy', 'pipeline_loaded': True, 'version': '1.0.0'}


In [19]:
# ── All endpoint tests ────────────────────────────────────────────────────────
import requests
import json

BASE_URL = "http://localhost:8000"

# ── TEST 1: Health ────────────────────────────────────────────────────────────
print("=" * 55)
print("TEST 1: GET /health")
print("=" * 55)
r = requests.get(f"{BASE_URL}/health")
print(f"Status : {r.status_code}")
print(f"Body   : {json.dumps(r.json(), indent=2)}")

# ── TEST 2: POST /chat — new session ─────────────────────────────────────────
print("\n" + "=" * 55)
print("TEST 2: POST /chat — Turn 1 (new session)")
print("=" * 55)
r = requests.post(f"{BASE_URL}/chat",
                  json={"question": "What is the punishment for murder under BNS 2023?"})
print(f"Status     : {r.status_code}")
data = r.json()
session_id = data["session_id"]
print(f"Session ID : {session_id}")
print(f"Answer     : {data['answer'][:120]}...")
print(f"Sources    : {len(data['sources'])} chunks")
for i, s in enumerate(data["sources"]):
    print(f"  [{i+1}] {s['source']} — relevance: {s['relevance_score']}")

# ── TEST 3: POST /chat — same session (memory test) ──────────────────────────
print("\n" + "=" * 55)
print("TEST 3: POST /chat — Turn 2 (same session = memory)")
print("=" * 55)
r = requests.post(f"{BASE_URL}/chat",
                  json={"question": "What about attempt to murder?",
                        "session_id": session_id})
data2 = r.json()
print(f"Status     : {r.status_code}")
print(f"Session ID : {data2['session_id']}")
print(f"Same as T1 : {data2['session_id'] == session_id}")
print(f"Answer     : {data2['answer'][:120]}...")

# ── TEST 4: POST /predict ─────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("TEST 4: POST /predict — Sentiment Classification")
print("=" * 55)
for text in [
    "This movie was absolutely brilliant, I loved every minute.",
    "Terrible film, complete waste of time.",
]:
    r = requests.post(f"{BASE_URL}/predict", json={"text": text})
    res = r.json()
    print(f"\nInput : {text}")
    print(f"Label : {res['label']}  |  Confidence: {res['confidence']:.2%}")
    print(f"Model : {res['model_used']}")

# ── TEST 5: Validation — bad inputs ──────────────────────────────────────────
print("\n" + "=" * 55)
print("TEST 5: Validation — bad inputs caught automatically")
print("=" * 55)

r = requests.post(f"{BASE_URL}/chat", json={})
print(f"\n[A] Missing 'question' → Status: {r.status_code}")
print(f"    Error: {r.json()['detail'][0]['msg']}")

r = requests.post(f"{BASE_URL}/chat", json={"question": ""})
print(f"\n[B] Empty question → Status: {r.status_code}")
print(f"    Error: {r.json()['detail']}")

r = requests.post(f"{BASE_URL}/predict", json={"text": "hi"})
print(f"\n[C] Text too short → Status: {r.status_code}")
print(f"    Error: {r.json()['detail']}")

TEST 1: GET /health
Status : 200
Body   : {
  "status": "healthy",
  "pipeline_loaded": true,
  "version": "1.0.0"
}

TEST 2: POST /chat — Turn 1 (new session)
Status     : 200
Session ID : 50ff1ef9-9aa1-4794-881a-b390910de00d
Answer     : Based on the Bharatiya Nyaya Sanhita 2023, regarding 'What is the punishment for murder under BNS 2023?': [This is where...
Sources    : 2 chunks
  [1] BNS_2023.pdf — relevance: 0.91
  [2] BNS_2023.pdf — relevance: 0.87

TEST 3: POST /chat — Turn 2 (same session = memory)
Status     : 200
Session ID : 50ff1ef9-9aa1-4794-881a-b390910de00d
Same as T1 : True
Answer     : Based on the Bharatiya Nyaya Sanhita 2023, regarding 'What about attempt to murder?': [This is where your LangChain LCEL...

TEST 4: POST /predict — Sentiment Classification

Input : This movie was absolutely brilliant, I loved every minute.
Label : POSITIVE  |  Confidence: 94.00%
Model : faisalimam19/bert-imdb-sentiment

Input : Terrible film, complete waste of time.
Label : POSITIVE  

## Part 4: The Swagger UI — Your Free API Documentation

**Go to http://localhost:8000/docs in your browser.**

What you'll see:
- Every endpoint listed with its HTTP method
- Click "Try it out" → fill in the request body → Execute
- See the actual HTTP response, status code, and response schema
- All of this was generated *automatically* from your Pydantic models

This is what you show a hiring manager or a potential collaborator.
Not a notebook — a real, documented, interactive API.

**http://localhost:8000/redoc** — alternative docs, cleaner for reading.
**http://localhost:8000/openapi.json** — the raw schema (machine-readable).

## Part 5: How to Connect Your Actual RAG Pipeline

The `get_rag_answer()` function above uses mock data.
To connect your Phase 3 pipeline, replace it with:

```python
# At startup — load once
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain.memory import ConversationBufferMemory
import chromadb

# These are loaded ONCE when the server starts
chroma_client = chromadb.PersistentClient(path="./chroma_bns_db")
retriever = ... # your hybrid BM25 + semantic retriever
llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=os.environ["GROQ_API_KEY"])
chain = ... # your LCEL chain

# In the endpoint — called per request
def get_rag_answer(question, session_id):
    history = session_store.get(session_id, [])
    result = chain.invoke({"question": question, "chat_history": history})
    return result["answer"], result["sources"]
```

Key principle: the ChromaDB client, retriever, LLM, and chain are 
instantiated ONCE at startup. The endpoint function only calls `.invoke()`.

In [20]:
requirements = """fastapi>=0.110.0
uvicorn[standard]>=0.27.0
pydantic>=2.0.0
langchain>=0.1.0
langchain-groq>=0.1.0
langchain-chroma>=0.1.0
chromadb>=0.4.0
sentence-transformers>=2.2.2
rank-bm25>=0.2.2
python-dotenv>=1.0.0
"""

with open("app/requirements.txt", "w") as f:
    f.write(requirements)

print("✅ requirements.txt written")
print(requirements)

✅ requirements.txt written
fastapi>=0.110.0
uvicorn[standard]>=0.27.0
pydantic>=2.0.0
langchain>=0.1.0
langchain-groq>=0.1.0
langchain-chroma>=0.1.0
chromadb>=0.4.0
sentence-transformers>=2.2.2
rank-bm25>=0.2.2
python-dotenv>=1.0.0



## Day 25 Summary

**What we built:**
- A production-structured FastAPI app with 3 endpoints
- Pydantic request/response models — type-safe contracts
- Session management for multi-turn conversation context
- Proper HTTP error handling with HTTPException
- Auto-generated Swagger UI documentation

**Key numbers to remember:**
- `200` — success
- `422` — validation error (Pydantic caught bad input)
- `500` — internal server error (your pipeline crashed)

**The pattern you'll use forever:**
1. Define Pydantic models (what's valid in, what's valid out)
2. Load heavy resources at startup (not per request)
3. Keep endpoints thin — they call other functions, they don't do the work
4. Never expose raw exceptions — always HTTPException with a safe message

